In [1]:
# ==============================================
# Zomato Bangalore Restaurant Analysis
# Predictive Modeling & Customer Segmentation
# ==============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, classification_report
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# -------------------------------
# 1. Load and Clean Data
# -------------------------------
print("Loading data...")

# Note: The CSV has embedded commas and multiline reviews. 
# We'll parse it manually using Python's csv module for robustness.
import csv

rows = []
with open('zomato.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        # If the row has fewer columns than header, skip (corrupted lines)
        if len(row) < len(header):
            continue
        rows.append(row)

df = pd.DataFrame(rows, columns=header)

print(f"Initial shape: {df.shape}")

# Clean rating column: extract numeric part from strings like "4.1/5"
def clean_rate(x):
    if pd.isna(x):
        return np.nan
    match = re.search(r'(\d+\.?\d*)', str(x))
    return float(match.group(1)) if match else np.nan

df['rate'] = df['rate'].apply(clean_rate)

# Clean approx_cost: remove commas, handle 'NEW', convert to numeric
def clean_cost(x):
    if pd.isna(x):
        return np.nan
    x = str(x).replace(',', '')
    if x.isdigit():
        return int(x)
    else:
        return np.nan

df['cost'] = df['approx_cost(for two people)'].apply(clean_cost)

# Clean votes: convert to numeric
df['votes'] = pd.to_numeric(df['votes'], errors='coerce').fillna(0).astype(int)

# Drop rows with missing essential data
df = df.dropna(subset=['rate', 'cost'])
df = df[df['rate'] > 0]
df = df[df['cost'] > 0]

# Binary features
df['online_order'] = df['online_order'].map({'Yes': 1, 'No': 0})
df['book_table'] = df['book_table'].map({'Yes': 1, 'No': 0})

# Create rating categories for classification
def rating_cat(r):
    if r < 3:
        return 'Low'
    elif r < 4:
        return 'Medium'
    else:
        return 'High'

df['rating_category'] = df['rate'].apply(rating_cat)

# Extract primary cuisine (first one in list)
df['primary_cuisine'] = df['cuisines'].apply(lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unknown')

print(f"Cleaned shape: {df.shape}")
print(df[['name', 'rate', 'cost', 'votes', 'online_order', 'location', 'primary_cuisine']].head())

# -------------------------------
# 2. Exploratory Data Analysis (EDA)
# -------------------------------
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Rating distribution
plt.figure()
sns.histplot(df['rate'], bins=20, kde=True)
plt.title('Distribution of Restaurant Ratings')
plt.xlabel('Rating')
plt.savefig('rating_dist.png')
plt.close()

# Cost distribution (log transform)
plt.figure()
sns.histplot(np.log1p(df['cost']), bins=30, kde=True)
plt.title('Log(Cost for two) Distribution')
plt.xlabel('log(cost)')
plt.savefig('cost_dist.png')
plt.close()

# Online ordering vs rating
plt.figure()
sns.boxplot(x='online_order', y='rate', data=df)
plt.title('Rating by Online Order Availability')
plt.savefig('online_order_box.png')
plt.close()

# Top locations by average rating
top_locs = df.groupby('location')['rate'].agg(['mean', 'count']).sort_values('mean', ascending=False).head(10)
plt.figure()
sns.barplot(x=top_locs.index, y=top_locs['mean'])
plt.xticks(rotation=45)
plt.title('Top 10 Locations by Average Rating')
plt.tight_layout()
plt.savefig('top_locations.png')
plt.close()

# Cuisine analysis
cuisine_rating = df.groupby('primary_cuisine')['rate'].agg(['mean', 'count']).sort_values('mean', ascending=False).head(10)
plt.figure()
sns.barplot(x=cuisine_rating.index, y=cuisine_rating['mean'])
plt.xticks(rotation=45)
plt.title('Top 10 Cuisines by Average Rating')
plt.tight_layout()
plt.savefig('top_cuisines.png')
plt.close()

# Correlation matrix (numeric features only)
numeric_cols = ['rate', 'cost', 'votes', 'online_order', 'book_table']
corr = df[numeric_cols].corr()
plt.figure()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.savefig('corr_matrix.png')
plt.close()

# -------------------------------
# 3. Feature Engineering
# -------------------------------
# One-hot encode categorical features: location, primary_cuisine (keep top categories)
top_locations = df['location'].value_counts().head(15).index
df['location_top'] = df['location'].apply(lambda x: x if x in top_locations else 'Other')

top_cuisines = df['primary_cuisine'].value_counts().head(15).index
df['cuisine_top'] = df['primary_cuisine'].apply(lambda x: x if x in top_cuisines else 'Other')

# Prepare feature matrix X and target y
X = df[['cost', 'votes', 'online_order', 'book_table', 'location_top', 'cuisine_top']]
y_reg = df['rate']
y_clf = df['rating_category']

# Preprocessing pipeline
numeric_features = ['cost', 'votes']
categorical_features = ['online_order', 'book_table', 'location_top', 'cuisine_top']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# -------------------------------
# 4. Regression Model (Predict Rating)
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)

# Random Forest Regressor
rf_reg = Pipeline(steps=[('preprocessor', preprocessor),
                         ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])
rf_reg.fit(X_train, y_train)
y_pred_reg = rf_reg.predict(X_test)

print("\n=== Regression Results ===")
print(f"MAE: {mean_absolute_error(y_test, y_pred_reg):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_reg)):.3f}")
print(f"R²: {r2_score(y_test, y_pred_reg):.3f}")

# Feature importance from Random Forest (after one-hot encoding)
# Extract feature names
ohe = rf_reg.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
cat_features = ohe.get_feature_names_out(categorical_features)
feature_names = numeric_features + list(cat_features)
importances = rf_reg.named_steps['regressor'].feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(10)
plt.figure()
feat_imp.plot(kind='bar')
plt.title('Top 10 Feature Importances (Random Forest Regressor)')
plt.tight_layout()
plt.savefig('feature_importance_reg.png')
plt.close()

# -------------------------------
# 5. Classification Model (Rating Category)
# -------------------------------
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42)

rf_clf = Pipeline(steps=[('preprocessor', preprocessor),
                         ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])
rf_clf.fit(X_train_clf, y_train_clf)
y_pred_clf = rf_clf.predict(X_test_clf)

print("\n=== Classification Results ===")
print(f"Accuracy: {accuracy_score(y_test_clf, y_pred_clf):.3f}")
print(classification_report(y_test_clf, y_pred_clf))

# -------------------------------
# 6. Clustering (Restaurant Segmentation)
# -------------------------------
# Use numeric + binary features for clustering
cluster_features = ['cost', 'votes', 'rate', 'online_order', 'book_table']
X_cluster = df[cluster_features].copy()
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Determine optimal k using elbow method
inertia = []
k_range = range(2, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure()
plt.plot(k_range, inertia, 'bo-')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.savefig('elbow.png')
plt.close()

# Choose k=4 (example)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Analyze cluster characteristics
cluster_summary = df.groupby('cluster')[cluster_features].mean()
print("\n=== Cluster Centers (mean values) ===")
print(cluster_summary)

# Visualize clusters with PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
plt.figure()
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster'], cmap='viridis', alpha=0.6)
plt.title('Restaurant Clusters (PCA projection)')
plt.colorbar(scatter)
plt.savefig('clusters_pca.png')
plt.close()

# -------------------------------
# 7. Save results and final insights
# -------------------------------
df.to_csv('zomato_cleaned_with_clusters.csv', index=False)
print("\nCleaned dataset with cluster labels saved as 'zomato_cleaned_with_clusters.csv'")

# Summary of findings
print("\n=== Key Insights ===")
print(f"1. Average restaurant rating: {df['rate'].mean():.2f}")
print(f"2. Restaurants with online ordering: {df['online_order'].mean()*100:.1f}%")
print(f"3. Top location by avg rating: {top_locs.index[0]} ({top_locs['mean'].iloc[0]:.2f})")
print(f"4. Top cuisine by avg rating: {cuisine_rating.index[0]} ({cuisine_rating['mean'].iloc[0]:.2f})")
print("5. Regression R² score: {:.3f} – cost, votes, location, and cuisine explain some variance in ratings".format(r2_score(y_test, y_pred_reg)))
print("6. Classification accuracy: {:.3f} – good separation of low/medium/high rated restaurants".format(accuracy_score(y_test_clf, y_pred_clf)))
print("7. Four distinct restaurant clusters identified: e.g., budget low-rated, expensive high-rated, etc.")

Loading data...
Initial shape: (56252, 13)
Cleaned shape: (41418, 16)
                    name  rate   cost  votes  online_order      location  \
0                  Jalsa   4.1  800.0    775             1  Banashankari   
1         Spice Elephant   4.1  800.0    787             1  Banashankari   
2        San Churro Cafe   3.8  800.0    918             1  Banashankari   
3  Addhuri Udupi Bhojana   3.7  300.0     88             0  Banashankari   
4          Grand Village   3.8  600.0    166             0  Basavanagudi   

  primary_cuisine  
0    North Indian  
1         Chinese  
2            Cafe  
3    South Indian  
4    North Indian  

=== Regression Results ===
MAE: 0.061
RMSE: 0.136
R²: 0.907

=== Classification Results ===
Accuracy: 0.970
              precision    recall  f1-score   support

        High       0.97      0.97      0.97      2376
         Low       0.91      0.83      0.87       476
      Medium       0.97      0.98      0.98      5432

    accuracy              